# Model validasi wajah untuk absensi

Notebook ini **tidak melatih CNN dari nol**. YuNet mendeteksi satu wajah, SFace pralatih mengubahnya menjadi embedding, lalu foto enrollment tiap karyawan dirata-ratakan menjadi template. Ini lebih tepat untuk dataset kantor yang kecil.

## Siapkan dataset di Google Drive

```text
MyDrive/absen-face/dataset/
├── EMP001/       # minimal 8 foto orang yang sama
├── EMP002/
└── _unknown/     # opsional: orang yang tidak terdaftar
```

Gunakan ID stabil, bukan nama, sebagai nama folder. Ambil 10–15 foto per karyawan pada hari dan kondisi berbeda. Satu foto hanya boleh memuat satu wajah. Jangan memakai frame video yang nyaris identik karena membuat evaluasi terlihat lebih bagus dari kondisi nyata.

> Embedding wajah adalah data biometrik sensitif. Batasi akses Drive dan server, gunakan persetujuan karyawan, kebijakan retensi, enkripsi, serta mekanisme absensi alternatif. Face recognition **bukan liveness detection**; sebelum produksi tambahkan pemeriksaan kedipan/gerakan atau SDK anti-spoofing.

In [ ]:
!pip -q install "opencv-python-headless>=4.10,<5"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import json
import random
import urllib.request

import cv2 as cv
import numpy as np
from PIL import Image, ImageOps

DATASET_DIR = Path('/content/drive/MyDrive/absen-face/dataset')
OUTPUT_FILE = Path('/content/drive/MyDrive/absen-face/face_templates.json')
MODEL_DIR = Path('/content/face_models')
YUNET_MODEL = MODEL_DIR / 'face_detection_yunet_2023mar.onnx'
SFACE_MODEL = MODEL_DIR / 'face_recognition_sface_2021dec.onnx'
YUNET_URL = 'https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx'
SFACE_URL = 'https://github.com/opencv/opencv_zoo/raw/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx'
MIN_IMAGES = 8
TARGET_FAR = 0.01  # maksimum 1% false accept pada data kalibrasi
SEED = 42
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
print('OpenCV:', cv.__version__)

## Ekstraksi embedding

Foto ditolak bila tidak berisi tepat satu wajah. Normalisasi orientasi EXIF mencegah foto kamera HP terbaca miring.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for url, destination in [(YUNET_URL, YUNET_MODEL), (SFACE_URL, SFACE_MODEL)]:
    if not destination.exists():
        urllib.request.urlretrieve(url, destination)

detector = cv.FaceDetectorYN.create(str(YUNET_MODEL), '', (320, 320), 0.9, 0.3, 5000)
recognizer = cv.FaceRecognizerSF.create(str(SFACE_MODEL), '')

def image_files(folder):
    return sorted(p for p in folder.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)

def embedding_from_image(path):
    try:
        pil_image = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
        image = cv.cvtColor(np.asarray(pil_image), cv.COLOR_RGB2BGR)
        detector.setInputSize((image.shape[1], image.shape[0]))
        _, faces = detector.detect(image)
        face_count = 0 if faces is None else len(faces)
        if face_count != 1:
            return None, f'wajah valid terdeteksi: {face_count}'
        aligned = recognizer.alignCrop(image, faces[0])
        vector = recognizer.feature(aligned).flatten().astype(np.float32)
        vector /= np.linalg.norm(vector)
        return vector, None
    except Exception as error:
        return None, str(error)

assert DATASET_DIR.exists(), f'Dataset tidak ditemukan: {DATASET_DIR}'
employee_dirs = sorted(p for p in DATASET_DIR.iterdir() if p.is_dir() and p.name != '_unknown')
assert len(employee_dirs) >= 2, 'Butuh minimal 2 folder karyawan untuk kalibrasi.'
print('Karyawan ditemukan:', [p.name for p in employee_dirs])

In [ ]:
vectors = defaultdict(list)
rejected = []

for folder in employee_dirs:
    for path in image_files(folder):
        vector, reason = embedding_from_image(path)
        if vector is None:
            rejected.append((str(path), reason))
        else:
            vectors[folder.name].append(vector)
    print(f'{folder.name}: {len(vectors[folder.name])} foto valid')

too_small = {
    folder.name: len(vectors[folder.name])
    for folder in employee_dirs
    if len(vectors[folder.name]) < MIN_IMAGES
}
assert not too_small, f'Foto valid kurang dari {MIN_IMAGES}: {too_small}'
print(f'Ditolak: {len(rejected)} foto')
for item in rejected[:20]:
    print(' -', item)

## Enrollment, kalibrasi, dan test

Setiap identitas dibagi deterministik menjadi sekitar 60% enrollment, 20% kalibrasi, dan 20% test. Threshold dipilih hanya dari bagian kalibrasi, lalu dilaporkan pada bagian test.

- **FAR**: orang salah diterima sebagai karyawan yang diklaim (semakin kecil semakin aman).
- **FRR**: karyawan benar ditolak (semakin kecil semakin nyaman).
- Threshold dipilih untuk memenuhi `TARGET_FAR`; ubah target hanya berdasarkan hasil uji lapangan.

In [ ]:
rng = random.Random(SEED)
splits = {}
templates = {}

for employee_id, items in sorted(vectors.items()):
    items = list(items)
    rng.shuffle(items)
    n = len(items)
    enrollment_end = max(4, int(n * 0.6))
    calibration_end = enrollment_end + max(2, int(n * 0.2))
    calibration_end = min(calibration_end, n - 2)
    splits[employee_id] = {
        'enrollment': items[:enrollment_end],
        'calibration': items[enrollment_end:calibration_end],
        'test': items[calibration_end:],
    }
    centroid = np.mean(splits[employee_id]['enrollment'], axis=0)
    templates[employee_id] = centroid / np.linalg.norm(centroid)
    counts = {part: len(values) for part, values in splits[employee_id].items()}
    print(employee_id, counts)

def similarity(vector, template):
    return float(np.dot(vector, template))  # keduanya sudah L2-normalized

def scores_for(part):
    genuine, impostor = [], []
    for real_id, employee_split in splits.items():
        for vector in employee_split[part]:
            genuine.append(similarity(vector, templates[real_id]))
            impostor.extend(
                similarity(vector, template)
                for claimed_id, template in templates.items()
                if claimed_id != real_id
            )
    return np.asarray(genuine), np.asarray(impostor)

calibration_positive, calibration_negative = scores_for('calibration')
test_positive, test_negative = scores_for('test')

# Foto _unknown menambah contoh negatif jika tersedia.
unknown_vectors = []
unknown_dir = DATASET_DIR / '_unknown'
if unknown_dir.exists():
    for path in image_files(unknown_dir):
        vector, reason = embedding_from_image(path)
        if vector is not None:
            unknown_vectors.append(vector)
        else:
            rejected.append((str(path), reason))
    rng.shuffle(unknown_vectors)
    middle = len(unknown_vectors) // 2
    for vector in unknown_vectors[:middle]:
        calibration_negative = np.append(
            calibration_negative,
            [similarity(vector, template) for template in templates.values()]
        )
    for vector in unknown_vectors[middle:]:
        test_negative = np.append(
            test_negative,
            [similarity(vector, template) for template in templates.values()]
        )
print('Skor kalibrasi:', len(calibration_positive), 'positif,', len(calibration_negative), 'negatif')
print('Skor test:', len(test_positive), 'positif,', len(test_negative), 'negatif')

In [ ]:
def rates(threshold, positive, negative):
    far = float(np.mean(negative >= threshold))
    frr = float(np.mean(positive < threshold))
    return far, frr

candidates = np.unique(np.concatenate([calibration_positive, calibration_negative]))
candidates = np.append(candidates, candidates.max() + 1e-6)
eligible = []
for threshold_candidate in candidates:
    far, frr = rates(threshold_candidate, calibration_positive, calibration_negative)
    if far <= TARGET_FAR:
        eligible.append((frr, far, float(threshold_candidate)))
assert eligible, 'Threshold tidak dapat dihitung.'
calibration_frr, calibration_far, threshold = min(eligible)
test_far, test_frr = rates(threshold, test_positive, test_negative)

print(f'Threshold cosine : {threshold:.4f}')
print(f'Kalibrasi FAR    : {calibration_far:.2%}')
print(f'Kalibrasi FRR    : {calibration_frr:.2%}')
print(f'Test FAR         : {test_far:.2%}')
print(f'Test FRR         : {test_frr:.2%}')
if test_far > TARGET_FAR:
    print('PERINGATAN: FAR test melewati target; tambah data negatif dan variasi kondisi.')

## Ekspor template

File JSON ini bukan model SFace; isinya template karyawan dan threshold hasil kalibrasi. Server inference tetap harus memakai pipeline YuNet + SFace yang sama. Jangan taruh file ini di aplikasi mobile atau repository publik.

In [ ]:
bundle = {
    'version': 1,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'backbone': 'opencv-zoo/face_recognition_sface_2021dec',
    'detector': 'opencv-zoo/face_detection_yunet_2023mar',
    'metric': 'cosine_similarity',
    'threshold': round(threshold, 7),
    'target_far': TARGET_FAR,
    'test_far': round(test_far, 7),
    'test_frr': round(test_frr, 7),
    'templates': {
        employee_id: [round(float(value), 7) for value in template]
        for employee_id, template in templates.items()
    },
}
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE.write_text(json.dumps(bundle, separators=(',', ':')), encoding='utf-8')
print('Tersimpan:', OUTPUT_FILE)
print('Jumlah template:', len(bundle['templates']))

## Uji satu foto

Masukkan ID yang diklaim (sama dengan nama folder), lalu upload satu selfie. Hasil `DITERIMA` hanya menguji kemiripan wajah—belum membuktikan bahwa selfie berasal dari orang hidup.

In [ ]:
from google.colab import files

claimed_id = input('ID karyawan yang diklaim: ').strip()
assert claimed_id in templates, f'ID tidak terdaftar: {claimed_id}'
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload tepat satu foto.'
photo_path = Path('/content') / next(iter(uploaded))
photo_path.write_bytes(next(iter(uploaded.values())))
probe, reason = embedding_from_image(photo_path)
assert probe is not None, f'Foto ditolak: {reason}'
score = similarity(probe, templates[claimed_id])
print(f'Score: {score:.4f} | Threshold: {threshold:.4f}')
print('DITERIMA' if score >= threshold else 'DITOLAK')

## Kontrak integrasi yang disarankan

Aplikasi saat ini sudah mengetahui identitas dari login, jadi lakukan verifikasi **1:1**: aplikasi mengirim selfie dan jenis absensi; server mencari template ID login, memastikan tepat satu wajah dan kualitas cukup, menghitung cosine similarity, lalu hanya mencatat absensi jika score ≥ threshold. Jangan percaya `employee_name`, `camera_access_granted`, score, atau keputusan cocok yang dikirim klien.

Urutan minimum sebelum dipakai sungguhan: autentikasi user di server → liveness/anti-spoof → ekstraksi embedding di server → cocokkan dengan template user login → cek lokasi/perangkat/jadwal → simpan score, versi model, dan hasil tanpa menyimpan selfie lebih lama dari kebijakan retensi.